# 01 — Data & Preprocessing for CryptoGraph-Secure

**Data.** 20 coins over 10 years (2016-09-28 → 2026-09-27). Yahoo Finance is the main source, cross-checked against Coin Metrics.
LUNC (Terra) and FTT (FTX) are kept as well, because the Terra and FTX crises started there.

*Why not CoinGecko?* Its free plan only serves the last 365 days of history.

This notebook reads the outputs of the data pipeline (`cryptograph-data --stage all`) and checks that they are fit for the deep-learning models.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"

from cryptograph.data.dataloader import load_processed_dataset

data = load_processed_dataset("../data/processed/crypto_graph_dataset.pt")   # graph-model tensors (20 coins)
panel = torch.load("../data/processed/v1/panel.pt", weights_only=False)        # features/targets for all coins
master = pd.read_parquet("../data/interim/master.parquet")                     # cleaned daily prices
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
audit = pd.read_csv("../reports/universe_audit.csv")
quality = pd.read_csv("../reports/data_quality_report.csv").set_index("coin")
balance = pd.read_csv("../reports/label_balance.csv")
counts = pd.read_csv("../reports/sample_counts.csv")
splits = json.loads(Path("../data/processed/v1/splits.json").read_text())["splits"]

asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])
close = master["close"].unstack("coin")
log_returns_df = np.log(close).diff()
panel_a = [c for c, a in zip(panel["coins"], panel["panel_A"]) if a]
day_split = np.array(panel["day_split"])

def add_events(fig):
    for _, e in events.iterrows():
        fig.add_vline(x=e["date"], line_width=1, line_dash="dash", line_color="red", opacity=0.7)
        fig.add_annotation(x=e["date"], y=1, yref="paper", text=e["name"].split("/")[0].strip(), showarrow=False,
                           textangle=-90, xanchor="right", yanchor="top", font=dict(color="red", size=9))
    return fig

coins_tbl = audit[audit["coin"].isin(asset_names + ["LUNC", "FTT"])][["coin", "nonstable_rank", "market_cap_usd", "yahoo_ticker", "is_event_coin"]]
coins_tbl = coins_tbl.assign(market_cap_usd=(coins_tbl["market_cap_usd"] / 1e9).round(1)).rename(
    columns={"nonstable_rank": "rank today", "market_cap_usd": "market cap $B", "yahoo_ticker": "Yahoo ticker", "is_event_coin": "event coin"})
print(f"{len(asset_names)} model coins + event coins LUNC, FTT; {len(timestamps)} days ({timestamps[0].date()} to {timestamps[-1].date()})")
coins_tbl.reset_index(drop=True)

20 model coins + event coins LUNC, FTT; 3652 days (2016-09-28 to 2026-09-27)


,coin,rank today,market cap $B,Yahoo ticker,event coin
0,BTC,1,1666.1,BTC-USD,False
1,ETH,2,323.3,ETH-USD,False
2,BNB,3,101.6,BNB-USD,False
3,XRP,4,93.0,XRP-USD,False
4,SOL,5,69.6,SOL-USD,False
5,TRX,6,31.7,TRX-USD,False
6,ZEC,7,26.4,ZEC-USD,False
7,DOGE,9,14.5,DOGE-USD,False
8,LINK,10,10.3,LINK-USD,False
9,XMR,11,10.0,XMR-USD,False


*What this shows:* Loads the pipeline outputs: the graph-model tensors for the 20 model coins, the feature/target panel, the cleaned daily prices and the verified crisis dates. The table lists the 20 coins (today's top 20 non-stable coins with enough history and liquidity) plus the two event coins, LUNC and FTT, kept because the Terra and FTX crises started there.

## 1. Coverage

Most coins only have full Yahoo data from **2017-11-09**. Before that there are only closing prices, from Coin Metrics.

*Is it really 10 years?* 10 years of closing prices; about 9 years of full open/high/low/close/volume.

In [2]:
src = master["close_source"].unstack("coin")
rows = []
for coin in asset_names + ["LUNC", "FTT"]:
    for label, key in (("Coin Metrics (close only)", "coinmetrics"), ("Yahoo (full OHLCV)", "yahoo")):
        d = src.index[src[coin] == key]
        if len(d):
            rows.append({"coin": coin, "source": label, "start": d.min(),
                         "end": (timestamps[-1] if key == "yahoo" else d.max()) + pd.Timedelta(days=1)})
fig = px.timeline(pd.DataFrame(rows), x_start="start", x_end="end", y="coin", color="source",
                  category_orders={"coin": asset_names + ["LUNC", "FTT"]},
                  color_discrete_map={"Coin Metrics (close only)": "#80cbc4", "Yahoo (full OHLCV)": "#1e88e5"},
                  title="Data coverage per coin and source")
fig.add_vline(x=pd.Timestamp("2017-11-09"), line_dash="dash", line_color="orange")
fig.update_layout(height=600, xaxis_range=[timestamps[0], timestamps[-1]], legend=dict(orientation="h", y=-0.08))
fig.show()

*What this shows:* Each bar is one coin. **Dark blue** = full Yahoo data (open, high, low, close, volume); **teal** = only a closing price from Coin Metrics; the orange line is 2017-11-09. Newer coins (SOL, AVAX, NEAR, WBT, …) simply start later. The models handle that with an `active_mask` instead of inventing earlier prices.

## 2. Quality

- **28 bad ticks** were corrected.
- **123 of the 156** very large daily moves (>40%) were confirmed by Coin Metrics or a known event; the rest are kept and flagged.
- **One AVAX gap** (69 days) was left missing and inactive, not filled.
- The two sources disagree on **23% of days in 2017** and about **0.2% from 2022**, so the early years are the least reliable.

In [3]:
q = quality.loc["ALL"]
display(pd.DataFrame({"coin-days": {
    "active coin-days": q["active_days"], "bad ticks corrected": q["bad_tick_corrected"],
    "very large moves (>40%)": q["large_return"], "... confirmed": q["large_return_confirmed"],
    "missing days (left inactive)": q["missing_day"], "gaps forward-filled": q["filled"],
    "high < low / zero price": q["high_lt_low"] + q["nonpositive_price"]}}).astype(int))

rec = pd.read_parquet("../data/interim/reconciled.parquet").reset_index()
by_year = (rec[rec["xsrc_flag"]].groupby(rec["date"].dt.year).size()
           / rec[rec["xsrc_rel_diff"].notna()].groupby(rec["date"].dt.year).size()).fillna(0)
fig = px.bar(x=by_year.index.astype(str), y=by_year.values, text=[f"{v:.1%}" for v in by_year.values],
             labels={"x": "year", "y": "share of days"}, title="Days where Yahoo and Coin Metrics disagree by more than 2%")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

,coin-days
active coin-days,90643
bad ticks corrected,28
very large moves (>40%),156
... confirmed,123
missing days (left inactive),69
gaps forward-filled,0
high < low / zero price,0


*What this shows:* The table counts what the quality checks found over 90,643 coin-days. Nothing was structurally wrong (no high < low, no zero prices, no gaps needed filling), 28 bad prices were corrected, and the only missing stretch (AVAX, 69 days) was left empty. The bar chart shows the two sources disagreeing on 23% of days in 2017 and ~0.2% from 2022, so any result that depends on 2017–18 should be read with care.

## 3. Prices and crises

Prices move across several orders of magnitude and are **non-stationary**, so the models use daily **log returns**, not prices.
Every major crisis (red lines, dates verified in `data/external/events.csv`) hits all coins at once.

In [4]:
selected = ["BTC", "ETH", "SOL", "BNB", "DOGE", "ADA", "LINK", "AVAX"]
growth = close[selected].apply(lambda s: s / s.dropna().iloc[0])
fig = go.Figure([go.Scatter(x=growth.index, y=growth[a], mode="lines", name=a) for a in selected])
add_events(fig)
fig.update_layout(title="Growth of each coin from its first price (log scale) with major crises",
                  yaxis_type="log", yaxis_title="growth factor", height=600)
fig.show()

from statsmodels.tsa.stattools import adfuller
m20 = asset_names
p_price = [adfuller(np.log(close[a].dropna()), autolag="AIC")[1] for a in m20]
p_ret = [adfuller(log_returns_df[a].dropna(), autolag="AIC")[1] for a in m20]
print(f"ADF unit-root test: prices non-stationary (p>0.05) for {sum(p > 0.05 for p in p_price)}/20 coins; "
      f"returns stationary (p<0.01) for {sum(p < 0.01 for p in p_ret)}/20")

ADF unit-root test: prices non-stationary (p>0.05) for 15/20 coins; returns stationary (p<0.01) for 20/20


*What this shows:* Prices of different coins span several orders of magnitude and trend over time. The ADF test confirms this: prices are non-stationary for 15/20 coins, while daily returns are stationary for 20/20. That is why every model input is built from **returns and volatility**, never raw prices. The red lines show that crises hit all coins at the same time.

## 4. Fat tails

Median excess kurtosis is about **18** (a normal distribution has 0) and normality is rejected for **all 20** coins.
Extreme days are far more common than a normal model expects, which is why we **clip the inputs** and use a **Huber loss**.

In [5]:
r20 = log_returns_df[asset_names]
kurt = r20.kurt()
jb_rejected = sum(stats.jarque_bera(r20[a].dropna()).pvalue < 0.01 for a in asset_names)
print(f"median excess kurtosis {kurt.median():.1f}; normality rejected for {jb_rejected}/20 coins")

btc = r20["BTC"].dropna() * 100
grid = np.linspace(btc.min(), btc.max(), 300)
fig = go.Figure([go.Histogram(x=btc, histnorm="probability density", nbinsx=150, name="BTC daily returns"),
                 go.Scatter(x=grid, y=stats.norm.pdf(grid, btc.mean(), btc.std()), name="normal distribution", line=dict(color="red"))])
fig.update_layout(title="BTC daily returns vs a normal distribution (log y-axis shows the fat tails)",
                  xaxis_title="daily log return (%)", yaxis_type="log", height=420)
fig.show()

median excess kurtosis 18.2; normality rejected for 20/20 coins


*What this shows:* The blue histogram is BTC's daily returns and the red curve is a normal distribution with the same mean and spread. On the log scale the normal curve vanishes beyond about ±10%, but BTC has many days out there (worst: −37% on 2020-03-12). Median excess kurtosis is 18.2 and normality is rejected for 20/20 coins. A squared-error loss would be dominated by these few days, so we use **Huber loss** and clip inputs at ±5 after scaling.

## 5. What is predictable?

- **Return direction is basically random:** the chance that tomorrow has the same sign as today is **0.47**.
- **Volatility clusters:** the autocorrelation of squared returns is about **0.18**.

So the targets are **volatility and shocks, not price direction**.

In [6]:
from statsmodels.tsa.stattools import acf
train_r = r20.loc[timestamps[day_split == "train"]]
same_sign = (np.sign(train_r.shift(1)) == np.sign(train_r)).where(train_r.notna() & train_r.shift(1).notna()).stack().mean()
acf_r = pd.DataFrame({a: acf(r20[a].dropna(), nlags=10, fft=True)[1:] for a in asset_names}).mean(axis=1)
acf_v = pd.DataFrame({a: acf(r20[a].dropna() ** 2, nlags=10, fft=True)[1:] for a in asset_names}).mean(axis=1)
print(f"P(tomorrow's sign = today's sign) = {same_sign:.2f};  lag-1 autocorrelation: returns {acf_r.iloc[0]:+.3f}, squared returns {acf_v.iloc[0]:+.3f}")

lags = list(range(1, 11))
fig = go.Figure([go.Bar(x=lags, y=acf_r.values, name="returns (direction)"),
                 go.Bar(x=lags, y=acf_v.values, name="squared returns (volatility)")])
fig.update_layout(barmode="group", title="Does today predict tomorrow? Autocorrelation, average over 20 coins",
                  xaxis_title="lag (days)", yaxis_title="autocorrelation", height=400)
fig.show()

P(tomorrow's sign = today's sign) = 0.47;  lag-1 autocorrelation: returns -0.024, squared returns +0.184


*What this shows:* The first bars (returns) are about zero at every lag: yesterday's direction tells you nothing about today's, and the same sign repeats only 47% of the time. The second bars (squared returns) are clearly positive (0.18 at lag 1): calm days follow calm days and turbulent days follow turbulent days. So the models predict **volatility and shocks**; return direction is kept only as a sanity check.

## 6. Labels

- A **shock** is a day where |z| > 2.5, with z = return / trailing 30-day volatility (known before the day). About **4%** of days are shocks.
- **Contagion:** after a shock at coin *i*, the chance that coin *j* also shocks within 1–3 days jumps from **under 1%** to **6–14%**.
  That jump is the signal the graph model has to learn.
- Shocks are rare, so we use **weighted or focal loss** and report **AUPRC**.

In [7]:
sh = balance[balance["label"].str.startswith("shock_next")]
fig = px.bar(sh, x="split", y="rate", text=[f"{v:.1%}" for v in sh["rate"]], title="Shock label: share of positive days per split")
fig.update_layout(height=350, yaxis_tickformat=".0%")
fig.show()

ct = balance[balance["label"].str.startswith("contagion") & (balance["split"] == "train")].copy()
ct["k"] = "within " + ct["label"].str[-1] + " day(s)"
long = pd.concat([ct.assign(case="any coin pair", value=ct["rate"]),
                  ct.assign(case="after coin i shocks", value=ct["rate_given_source_shock"])])
fig = px.bar(long, x="k", y="value", color="case", barmode="group", text=[f"{v:.1%}" for v in long["value"]],
             title="Contagion (train): chance that coin j shocks, overall vs after a shock at coin i")
fig.update_layout(height=380, yaxis_tickformat=".0%", xaxis_title=None)
fig.show()

*What this shows:* The first chart shows the shock label is positive on ~4% of days in every split, so a model that always says "no shock" would be 96% accurate and useless. That is why we report **AUPRC** and use **weighted or focal loss**. The second chart is the key contagion evidence: for a random coin pair a follow-on shock is rare (<1%), but right after coin *i* shocks, coin *j* shocks within 1–3 days 6–14% of the time. The graph model has to learn that jump.

## 7. Scaling

Robust scaling (median / IQR) **per coin**, fitted on **training dates only**. Volumes differ by orders of magnitude between coins,
so this makes the network learn *"abnormal for this coin"* rather than *which coin it is*.

In [8]:
fi = panel["feature_names"].index("log_volume")
rows = []
for coin in ["BTC", "ETH", "DOGE", "LINK", "HBAR"]:
    j = panel["coins"].index(coin)
    ok = panel["X_mask"][:, j, fi].numpy()
    for label, arr in (("raw log volume", panel["X_raw"]), ("after per-coin scaling", panel["X"])):
        rows.append(pd.DataFrame({"value": arr[:, j, fi].numpy()[ok], "coin": coin, "version": label}))
fig = px.box(pd.concat(rows), x="coin", y="value", color="coin", facet_col="version", points=False,
             title="Log volume before and after scaling")
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=420, showlegend=False)
fig.show()

*What this shows:* Left: raw log volume. BTC and ETH trade far more than HBAR or LINK, so the boxes sit at very different levels. Right: after per-coin robust scaling (median/IQR fitted on training dates only) all coins are centred on 0 with similar spread. The network now sees *"unusually high volume for this coin"* instead of just learning which coin is which, and no test-period information leaks into the scaler.

## 8. Splits and distribution shift

- **Train** runs to mid-2022, **validation** through 2023, **test** from 2024, with a **30-day gap** at the start of validation and test.
  Samples whose input or target window crosses a boundary are dropped.
- Validation and test are about **40% calmer** than train, but the **shock rate stays about 4%** in every split,
  so scale-free targets (z-scores, volatility ratios) are the right choice.

In [9]:
display(pd.DataFrame(splits).T[["start", "end", "embargo_end", "first_sample_date", "last_sample_date"]])

rv30 = log_returns_df[asset_names].rolling(30).std().median(axis=1) * np.sqrt(365) * 100
fig = px.line(x=rv30.index, y=rv30.values, labels={"x": "", "y": "annualised volatility (%)"},
              title="Typical coin volatility (median of 20 coins, 30-day) with train / validation / test")
for name, col in (("train", "royalblue"), ("val", "orange"), ("test", "green")):
    fig.add_vrect(x0=splits[name]["start"], x1=splits[name]["end"], fillcolor=col, opacity=0.12, line_width=0,
                  annotation_text=name, annotation_position="top left")
fig.update_layout(height=400)
fig.show()

shock_rate = balance[balance["label"].str.startswith("shock_next")].set_index("split")["rate"]
display(pd.DataFrame({"mean volatility %": [rv30[day_split == s].mean() for s in ("train", "val", "test")],
                      "shock rate": [shock_rate[s] for s in ("train", "val", "test")]},
                     index=["train", "val", "test"]).style.format({"mean volatility %": "{:.0f}", "shock rate": "{:.1%}"}))

,start,end,embargo_end,first_sample_date,last_sample_date
train,2016-09-28,2022-06-30,None,2016-10-27,2022-05-31
val,2022-07-01,2023-12-31,2022-07-30,2022-07-31,2023-12-01
test,2024-01-01,2026-09-27,2024-01-30,2024-01-31,2026-08-28


,mean volatility %,shock rate
train,107,3.9%
val,63,4.1%
test,72,3.9%


*What this shows:* The table gives the split dates. The 30-day embargo means the first usable validation/test sample is 30 days after the boundary, and the last sample of each split is 30 days before its end, so no input or target window crosses a boundary. The chart shows training (2016–2022) is far more volatile than validation and test, yet the shock rate stays ~4% everywhere. Raw volatility levels shift but standardised targets don't, which is why the targets are scale-free.

## 9. Correlation

Average pairwise correlation is **0.57**, and it rises after crises (**0.61 vs 0.54**).
A large part of each coin's movement is shared with the others, which a per-coin model cannot see: evidence that a **graph model** is needed.

In [10]:
rA = log_returns_df.loc["2017-11-09":, panel_a].dropna()
corrA = rA.corr()
off = corrA.values[~np.eye(len(corrA), dtype=bool)]
fig = px.imshow(corrA, color_continuous_scale="Viridis", zmin=0, zmax=1, labels=dict(color="Correlation"),
                title=f"Pairwise return correlation, 20 coins with full data from 2017-11-09 (average {off.mean():.2f})")
fig.update_layout(width=700, height=650)
fig.show()

from cryptograph.analysis.dependence import average_pairwise_corr
avgc = average_pairwise_corr(log_returns_df[asset_names], window=60, min_obs=40)
after = pd.Series(False, index=avgc.index)
for d in events["date"]:
    after.loc[d: d + pd.Timedelta(days=30)] = True
print(f"average correlation in the 30 days after a crisis: {avgc[after].mean():.2f} vs other days: {avgc[~after].mean():.2f}")
fig = add_events(px.line(x=avgc.index, y=avgc.values, labels={"x": "", "y": "average pairwise correlation"},
                         title="How strongly the 20 coins move together (60-day rolling)"))
fig.update_layout(height=420)
fig.show()

average correlation in the 30 days after a crisis: 0.61 vs other days: 0.54


*What this shows:* The heatmap shows every pair of coins is positively correlated (0.39–0.80, average 0.57): a large part of each coin's daily move is shared with the others. The line shows this co-movement changes over time and rises after crises (0.61 in the 30 days after an event vs 0.54 otherwise). A per-coin LSTM cannot use this shared information; a graph model can.

## 10. Edges over time

The graph changes every day, which justifies a **temporal** graph model.

- **Edge threshold 0.6:** lower thresholds connect almost every pair of coins, which is useless.
- **Direction comes from time order** (a coin that shocks first sends events to the others), because measured lag-1 direction was no better than noise.

In [11]:
edge_series = pd.Series([e.size(1) // 2 for e in data["edge_indices"]], index=timestamps)
n_active = data["active_mask"].numpy().sum(axis=1)
fig = go.Figure([
    go.Scatter(x=edge_series.index, y=edge_series, mode="lines", name="edges (coin pairs)", line=dict(color="#00e676", width=1)),
    go.Scatter(x=edge_series.index, y=edge_series.rolling(30).mean(), mode="lines", name="30-day average", line=dict(color="#ffea00", width=2)),
    go.Scatter(x=edge_series.index, y=n_active * (n_active - 1) / 2, mode="lines", name="all possible pairs", line=dict(color="grey", dash="dot"))])
fig.update_layout(title="Daily graph: coin pairs with 30-day |correlation| ≥ 0.6", yaxis_title="edges", height=420)
fig.show()

from cryptograph.data.graphs.correlation import rolling_corr_mats
C, D = rolling_corr_mats(log_returns_df[asset_names].reindex(timestamps).to_numpy(), 30, 20)
iu = ~np.eye(len(asset_names), dtype=bool)
dens = pd.DataFrame([{"threshold": t, "split": s, "share of pairs connected": np.nanmean(np.abs(C[day_split == s][:, iu]) >= t)}
                     for t in (0.3, 0.4, 0.5, 0.6, 0.7) for s in ("train", "val", "test")])
fig = px.line(dens, x="threshold", y="share of pairs connected", color="split", markers=True,
              title="How many coin pairs get an edge at each threshold (0.6 chosen)")
fig.add_vline(x=0.6, line_dash="dash", line_color="red")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

chance = 1 - stats.norm.cdf(np.arctanh(0.3) * np.sqrt(27))
found = {s: np.nanmean(D[day_split == s][:, iu] > 0.3) for s in ("train", "val", "test")}
print("Directed 'coin i yesterday -> coin j today' edges (lag-1 corr > 0.3): found "
      + ", ".join(f"{s} {v:.1%}" for s, v in found.items()) + f"; expected by pure chance {chance:.1%}")

Directed 'coin i yesterday -> coin j today' edges (lag-1 corr > 0.3): found train 1.6%, val 3.6%, test 4.3%; expected by pure chance 5.4%


*What this shows:* The first chart is the daily graph fed to the graph models: an edge joins two coins when their last 30 days of returns have |correlation| ≥ 0.6. The edge count changes every day and grows as new coins are listed (grey line = all possible pairs), so the model must be **temporal**. The second chart shows why 0.6: at 0.3–0.4 almost every pair is connected in val/test, which carries no structure. The printed line shows that 'coin i yesterday → coin j today' edges appear no more often than pure chance (1.6–4.3% vs 5.4%). So the TGN gets its direction from **which coin shocked first**, not from a noisy correlation estimate.

## 11. Tensor check

Shapes and sample counts for each model, loaded through the real DataLoaders. It is **small data by deep-learning standards**
(about 2,000 training days), so we use **small models, dropout and early stopping**.

In [12]:
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PygLoader
from cryptograph.data.datasets import LSTMWindowDataset, GraphSnapshotDataset, load_event_stream

rows = []
for s in ("train", "val", "test"):
    lstm = LSTMWindowDataset(f"../data/processed/v1/lstm_{s}.pt")
    b = next(iter(DataLoader(lstm, batch_size=64)))
    rows.append({"model": "LSTM windows", "split": s, "samples": len(lstm), "one batch": f"x {tuple(b['x'].shape)}, y {tuple(b['y_ret'].shape)}"})
    snap = GraphSnapshotDataset("../data/processed/v1/graph_snapshots.pt", s)
    g = next(iter(PygLoader(snap, batch_size=8)))
    rows.append({"model": "graph snapshots", "split": s, "samples": len(snap), "one batch": f"x {tuple(g.x.shape)}, edges {tuple(g.edge_index.shape)}"})
ev = load_event_stream("../data/processed/v1/tgn_events.pt")
for s in ("train", "val", "test"):
    rows.append({"model": "TGN events", "split": s, "samples": ev[s].num_events,
                 "one batch": f"src/dst/t {tuple(ev[s].src[:200].shape)}, msg {tuple(ev[s].msg[:200].shape)}"})
tensors = pd.DataFrame(rows)
display(tensors.pivot(index="model", columns="split", values="samples")[["train", "val", "test"]])
display(tensors[tensors["split"] == "train"][["model", "one batch"]].reset_index(drop=True))
print(f"Inputs contain no NaN/inf: {bool(torch.isfinite(b['x']).all() and torch.isfinite(g.x).all() and torch.isfinite(ev['train'].msg).all())}; "
      f"event times sorted: {bool((ev['full'].t[1:] >= ev['full'].t[:-1]).all())}")

split,train,val,test
model,,,
LSTM windows,28880,9725,18820
TGN events,16266,8241,14877
graph snapshots,2043,489,941


,model,one batch
0,LSTM windows,"x (64, 30, 12), y (64, 5)"
1,graph snapshots,"x (160, 12), edges (2, 118)"
2,TGN events,"src/dst/t (200,), msg (200, 5)"


Inputs contain no NaN/inf: True; event times sorted: True


*What this shows:* Each model's data loads through its real DataLoader. The LSTM gets 30-day windows of 12 features per coin; the graph models get one graph per day (batches of 8 days × 20 coins = 160 nodes); the TGN gets a time-sorted stream of shock events with 5 message features. There are no NaN or infinite values and event times are in order. With only ~2,000 training days the dataset is small for deep learning, so models must be small, regularised (dropout, weight decay) and early-stopped on validation.

**Next:** `02` trains the LSTM baseline and the TGN on these tensors and compares them with simple baselines (`reports/naive_baselines.csv`);
`03_statistical_spillover_baseline.ipynb` holds the Diebold–Yilmaz ranking that TempME is validated against.
A longer optional analysis is in `01_dpeda.ipynb`.